# 2.03 Cohere Embeddings

Cohere의 `embed-multilingual-v3.0`은 **100+ 언어를 단일 모델로** 임베딩한다. 한국어 품질이 공개 벤치마크 상위권이며, `input_type` 파라미터로 비대칭 인덱싱을 지원한다.
또한 `embedding_types`로 **int8/binary 양자화 벡터**를 직접 받아 저장 비용을 크게 줄일 수 있다.

## 학습 목표

- `CohereEmbeddings`로 쿼리·문서를 벡터화한다
- `embed-multilingual-v3.0`의 `input_type`을 전환해 검색 품질을 높인다
- `embedding_types`로 float 대신 int8/binary 벡터를 받는다
- Cohere Rerank(05_retrievers에서 상세)와 함께 쓰는 2단계 파이프라인의 그림을 익힌다

## 언제 쓰나

- **한국어·일본어·아랍어 등 다국어 RAG**에서 공급자 단일화가 필요할 때
- **대규모 인덱스의 저장 비용**을 줄이려고 int8/binary 양자화가 필요할 때
- **Rerank**를 같은 공급자로 묶어 레이턴시와 계약을 단순화하고 싶을 때

## 2.03.1 환경 설정

필요 패키지: `langchain-cohere`. `.env`에 `COHERE_API_KEY`가 있어야 한다.

In [ ]:
# !pip install -U langchain langchain-cohere

import os
from dotenv import load_dotenv
load_dotenv(override=True)

assert os.environ.get("COHERE_API_KEY"), "COHERE_API_KEY 필요"

## 2.03.2 기본 사용법

다국어 모델을 기본으로 사용한다.

In [ ]:
from langchain_cohere import CohereEmbeddings

embeddings = CohereEmbeddings(model="embed-multilingual-v3.0")

q_vec = embeddings.embed_query("Cohere 다국어 임베딩이란?")
print("query dim:", len(q_vec), "| preview:", q_vec[:4])

docs = [
    "Cohere embed-multilingual-v3.0은 1024차원을 생성한다.",
    "한국어, 영어, 일본어를 한 공간에서 인덱싱한다.",
    "int8/binary 양자화로 저장 비용을 줄일 수 있다.",
]
doc_vecs = embeddings.embed_documents(docs)
print("docs:", len(doc_vecs), "x", len(doc_vecs[0]))

## 2.03.3 차원 · 비용 · 다국어 성능

| 모델 | 차원 | 언어 | 용도 |
|------|------|------|------|
| `embed-multilingual-v3.0` | 1024 | 100+ | 다국어 RAG (권장) |
| `embed-english-v3.0` | 1024 | 영어 | 영어 전용 최고 품질 |
| `embed-multilingual-light-v3.0` | 384 | 100+ | 저지연/저비용 |
| `embed-english-light-v3.0` | 384 | 영어 | 저지연/저비용 |

- 최대 입력 토큰: 512 (초과 시 `truncate`로 처리)
- 다국어 v3 모델은 KLUE, MIRACL 한국어 서브셋에서 상위권
- 가격은 토큰 단위로 부과 — 공식 가격표 참고

### `truncate` 전략

512 토큰 초과 입력에 대해 `'NONE' | 'START' | 'END'` 중 선택. 기본은 `'END'`로 뒤를 잘라낸다.

In [ ]:
long_emb = CohereEmbeddings(
    model="embed-multilingual-v3.0",
    truncate="END",
)
long_text = "긴 문서 " * 400  # 일부러 토큰을 넘겨본다
v = long_emb.embed_query(long_text)
print("truncated query dim:", len(v))

## 2.03.4 벡터스토어 연동 미리보기

1024차원을 받을 수 있는 vector store에 붙인다. Chroma는 차원을 첫 임베딩으로 자동 추론한다.

In [ ]:
# !pip install -U langchain-chroma
from langchain_chroma import Chroma

store = Chroma.from_texts(
    texts=docs,
    embedding=CohereEmbeddings(model="embed-multilingual-v3.0"),
    collection_name="demo_cohere",
)
hits = store.similarity_search("저장 비용을 줄이는 양자화", k=2)
for h in hits:
    print("-", h.page_content)

## 2.03.5 `embedding_types` — 양자화 벡터

Cohere v3 모델은 여러 타입의 벡터를 동시에 반환할 수 있다.

| 타입 | 바이트/차원 | 용도 |
|------|------------|------|
| `float` | 4 | 기본, 최고 품질 |
| `int8` | 1 | 4배 압축, 품질 거의 유지 |
| `uint8` | 1 | int8의 unsigned 버전 |
| `binary` | 1/8 | 32배 압축, 해밍 거리 기반 검색용 |
| `ubinary` | 1/8 | binary의 unsigned |

기본 `embed_query`는 float를 반환하지만, client를 직접 호출하면 여러 타입을 한 번에 받을 수 있다.

In [ ]:
int8_emb = CohereEmbeddings(
    model="embed-multilingual-v3.0",
    embedding_types=["int8"],
)
# LangChain 래퍼는 첫 번째 타입의 벡터를 반환한다
v_int8 = int8_emb.embed_query("int8 양자화 벡터")
print("int8 dim:", len(v_int8), "| sample:", v_int8[:6])

## 2.03.6 Rerank와의 2단계 파이프라인

Cohere의 진짜 강점은 **임베딩 + Rerank 콤보**. 1단계로 임베딩 유사도 상위 k를 뽑고, 2단계로 `rerank-multilingual-v3.0`이 교차 주의 기반으로 재정렬한다.
상세 구현은 `../05_retrievers/`에서 다루고, 여기서는 개념만 확인한다.

In [ ]:
from langchain_cohere import CohereRerank

reranker = CohereRerank(model="rerank-multilingual-v3.0", top_n=2)

from langchain_core.documents import Document
candidates = [Document(page_content=t) for t in [
    "파이썬은 인터프리터 언어다.",
    "Cohere Rerank는 검색 결과를 재정렬한다.",
    "임베딩은 의미를 벡터로 압축한다.",
]]
ranked = reranker.compress_documents(candidates, query="검색 재정렬이 필요한 이유")
for d in ranked:
    print("-", d.page_content)

## 체크리스트

- [ ] 다국어 코퍼스라면 `embed-multilingual-v3.0` 기본값 확인
- [ ] 512 토큰 초과 입력에 `truncate` 전략 명시
- [ ] 저장 비용이 크다면 `embedding_types=['int8']` 또는 `['binary']` 검토
- [ ] Rerank를 함께 쓸 계획이면 같은 공급자로 계약 통일

## 다음

- `04_voyageai.ipynb` — Voyage 3의 RAG 특화 임베딩
- `../05_retrievers/` — Cohere Rerank를 리트리버 체인에 넣기

## 참고

- 공식: https://docs.langchain.com/oss/python/integrations/text_embedding/cohere
- Cohere Embed v3: https://docs.cohere.com/docs/embeddings
- int8/binary 양자화: https://docs.cohere.com/docs/int8-binary-embeddings